# 23. Lineups and minutes (simulator step 1)

The season simulator needs to know **who plays and how much**. We have every player's projected WAR/60 (notebooks 18-22) and his opening-night team and status (notebook 22). This notebook turns that into a lineup for each team: **12 forwards (4 lines), 6 defensemen (3 pairs), a starting goalie and a backup**, each with minutes per game.

**How the coach picks.** Within forwards, defensemen and goalies, every player gets a **coach score**:

> coach score = w x (standardized ice time per game last season) + (1 - w) x (standardized projected WAR/60)

- `w = 1`: the coach goes by who played the most last season. `w = 0`: he goes by our projection only.
- For goalies, "usage" is games per 82 last season instead of minutes (a goalie plays about 60 minutes every game he's in).
- "Standardized" (a z-score) means: minus the league average, divided by the spread, so minutes and WAR are on the same scale.
- A player with no usable NHL ice time last season (under 10 games: rookies, most call-ups) gets it from a rule, tested below.

**Minutes come from the slot, not the player.** The best forward on a team plays what a team's best forward typically plays, the 12th what a 12th forward plays. Goalies split games: the starter gets the typical starter's share.

**A backtest picks w**: for past seasons, using only what was known before each season, which w best predicts the ice time players actually got?

Then: all 32 opening-night lineups, a first look at Ryan's experiment (Detroit with and without Larkin and Edvinsson; Winnipeg with and without the Hellebuyck trade), and a check on Ben Chiarot.

Rules for this version: lineups are fixed once at the start (no hot and cold streaks), **full health** (no injuries), and the two contract disputes (Larkin, Edvinsson) **play by default**, with a sit-out switch. Code: `src/lineups.py`.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, project, backtest, aging_curve, group_average, current_projections, GOALS_PER_WIN
from edge_data import build_bio_table, first_nhl_teams, download_standings, standings_table
from nhle import load_profiles
from development import (training_nhle, best_nhle, skater_features, fit_prior, apply_prior, candidates, fit_rows,
                         experience, TARGETS, AGING, SK_WEIGHTS, SK_REG, G_WEIGHTS, G_REG, PRIOR_INPUTS)
import lineups as L

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
OUT = DATA                              # where lineups_2026-27.csv is saved
TRADES_PATH = DATA / "trades_2026-27.csv"
TODAY = "2026-11-06"                    # the standings date (the file is saved once, then read)

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

NAMES = {"F": "Forwards", "D": "Defensemen", "G": "Goalies"}

skater_war = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalie_war = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
history = history_table(skater_war, goalie_war)
skaters, goalies = history[history["group"] != "G"], history[history["group"] == "G"]
bios = build_bio_table(DATA / "bio_raw", 2026)
profiles = load_profiles(DATA / "bio_raw")
careers = pd.read_csv(DATA / "career_totals.csv")
rosters = pd.read_csv(DATA / "rosters_2026-27.csv")

TRAIN_END = 2017                         # same split as notebooks 18 and 21
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]
print("Train targets:", ", ".join(label(t) for t in TRAIN))
print("Test targets: ", ", ".join(label(t) for t in TEST))

Train targets: 2009-10, 2010-11, 2011-12, 2013-14, 2014-15, 2015-16, 2016-17, 2017-18
Test targets:  2018-19, 2021-22, 2022-23, 2023-24, 2024-25, 2025-26


## 2. The trade log: six more in-season moves

Notebook 22 found six players who played NHL games for two teams this season and weren't in the trade log yet. They're added here as new trades, **built from the saved game logs** (nothing typed in). Two moves are one trade when the teams swap within 7 days (Shane Wright and Matthew Wood).

- **The date** is each player's first game with his new team, so the trade happened on or before it.
- **Only missing trades are added.** Existing rows are never changed, and a rerun adds nothing.
- **The baseline still applies no trades.** These only matter when the simulator applies them.

**Heads-up:** this doesn't change `rosters_2026-27.csv`. If notebook 22 is ever rerun, these players' `rule` text changes from "first 2026-27 NHL team" to "trade log ... undone" and its counts shift; their teams and statuses stay the same.

In [2]:
trades = pd.read_csv(TRADES_PATH)
_, moves = first_nhl_teams(DATA / "current_raw" / "game_logs")
new = moves[~moves["playerId"].isin(trades["playerId"])].sort_values(["first_game", "playerId"]).reset_index(drop=True)
new["name"] = new["playerId"].map(rosters.set_index("playerId")["name"])

# Group into trades: a move joins an earlier one when the teams are swapped and the dates are within 7 days
new["trade_id"] = None
next_id = int(trades["trade_id"].str[1:].astype(int).max()) + 1
for i, m in new.iterrows():
    earlier = new.loc[:i - 1]
    match = earlier[(earlier["from_team"] == m["to_team"]) & (earlier["to_team"] == m["from_team"])
                    & ((pd.to_datetime(m["first_game"]) - pd.to_datetime(earlier["first_game"])).dt.days <= 7)]
    if len(match):
        new.at[i, "trade_id"] = match["trade_id"].iloc[0]
    else:
        new.at[i, "trade_id"] = f"T{next_id}"
        next_id += 1
new["date"] = new.groupby("trade_id")["first_game"].transform("min")
new["note"] = "from game logs (nb22 candidate): date = first game with the new team, so the trade was on or before it"
add = new[["trade_id", "date", "playerId", "name", "from_team", "to_team", "note"]]

In [3]:
SIX = ["Hallander", "Struble", "Hunt", "St. Ivany", "Shane Wright", "Matthew Wood"]   # notebook 22's candidates
def has(names, key):
    return any(key in n for n in names)

if len(add):
    assert len(add) == 6 and all(has(list(add["name"]), k) for k in SIX), add
    trades = pd.concat([trades, add], ignore_index=True)
    trades.to_csv(TRADES_PATH, index=False)
    print(f"Added {len(add)} rows ({add['trade_id'].nunique()} trades) to {TRADES_PATH.name}:")
else:
    print(f"Nothing to add: every in-season move is already in {TRADES_PATH.name}.")
assert all(has(list(trades["name"]), k) for k in SIX)
print(trades.drop(columns="note").to_string(index=False))

Added 6 rows (5 trades) to trades_2026-27.csv:
trade_id       date  playerId              name from_team to_team
      T1 2026-10-23   8476945 Connor Hellebuyck       WPG     NJD
      T1 2026-10-23   8482110     Dawson Mercer       NJD     WPG
      T1 2026-10-23   8482076         Nico Daws       NJD     WPG
      T1 2026-10-23   8475760     Nick Bjugstad       NJD     WPG
      T1 2026-10-23   8486030 Alexander Command       NJD     WPG
      T2 2026-10-10   8480842   Filip Hallander       PIT     EDM
      T3 2026-10-10   8481593    Jayden Struble       MTL     NSH
      T4 2026-10-25   8482094       Daemon Hunt       MIN     NSH
      T5 2026-10-28   8481030    Jack St. Ivany       WPG     WSH
      T6 2026-11-03   8483524      Shane Wright       SEA     NSH
      T6 2026-11-03   8484241      Matthew Wood       NSH     SEA


## 3. Minutes per slot, the goalies' split, and the rookie default

**The minutes table.** For each team-season, rank its players with 20+ games by ice time per game (all situations); then take the median by rank across teams. For 2026-27 it uses the three seasons before (2023-24 to 2025-26).

**Why it's scaled.** A team's dressed forwards share about 182 minutes a game and its defensemen about 117 (the league's total minutes / team games, overtime included). The raw top-12 add up to more, because when a regular is hurt his replacement also plays and also counts. With full health the same 12 play every game, so ranks 1-12 (and D 1-6) are **scaled to the real totals**: same shape, right total. The backtest uses the raw table, because it predicts the ice time of everyone who played 20+ games, injury fill-ins included.

**Goalies.** The starter's share of his team's top two goalies' games, every team-season. In v1 the backup gets the rest; back-to-backs come in the season step.

**Rookie default (rule R1).** The median ice time per game of true rookies (no NHL experience before that season, 10+ games), or for goalies their games per 82.

In [4]:
faceoffs = L.faceoff_rates([DATA / "skaters_2008_to_2024.csv", DATA / "skaters25-26.csv"])
usage = L.usage_table(skater_war, goalie_war, faceoffs)
exp = experience(history, profiles, history[["playerId", "season"]])

RECENT = [2023, 2024, 2025]
totals = L.team_minutes(usage, RECENT)
slots = L.scaled_slots(L.slot_minutes(usage, RECENT), totals)
print(f"A team's minutes per game, {label(RECENT[0])} to {label(RECENT[-1])}: forwards {totals['F']:.1f}, "
      f"defensemen {totals['D']:.1f} (raw top 12 F add up to {slots.loc['F'].loc[1:12, 'raw'].sum():.1f}, "
      f"top 6 D to {slots.loc['D'].loc[1:6, 'raw'].sum():.1f})")
print("\nMinutes per game by rank on the team (raw median; scaled = used for 2026-27 lineups):")
for g in ["F", "D"]:
    print(f"\n{NAMES[g]}")
    print(slots.loc[g].T.round(2).to_string())

A team's minutes per game, 2023-24 to 2025-26: forwards 180.6, defensemen 116.6 (raw top 12 F add up to 189.1, top 6 D to 120.3)

Minutes per game by rank on the team (raw median; scaled = used for 2026-27 lineups):

Forwards
rank       1      2      3      4      5      6      7      8      9      10     11     12     13     14    15    16
raw     20.19  19.24  18.34  17.48  16.78  16.12  15.41  14.58  13.82  12.95  12.44  11.77  10.99  10.21  9.59  9.04
scaled  19.28  18.38  17.52  16.69  16.03  15.40  14.72  13.93  13.20  12.37  11.88  11.24    NaN    NaN   NaN   NaN

Defensemen
rank        1      2      3      4      5      6      7      8      9
raw     24.16  22.11  20.65  19.33  17.68  16.34  14.99  14.21  13.21
scaled  23.43  21.44  20.02  18.74  17.14  15.84    NaN    NaN    NaN


In [5]:
shares, SHARE = L.starter_share(usage, RECENT)
by_season = shares.groupby(level="season").agg(["median", "mean", "min", "max", "count"])
by_season.index = by_season.index.map(label)
print("Starter's share of the top two goalies' games, by season:")
print(by_season.round(3).to_string())
print(f"\nUsed for 2026-27: starter {SHARE:.3f} ({SHARE * L.SEASON_GAMES_2026:.0f} of 84 games), "
      f"backup {1 - SHARE:.3f} ({(1 - SHARE) * L.SEASON_GAMES_2026:.0f})")

DEFAULTS = L.rookie_defaults(usage, exp, RECENT)
print(f"\nRookie default (R1), {label(RECENT[0])} to {label(RECENT[-1])}: forwards {DEFAULTS['F']:.2f} min/game, "
      f"defensemen {DEFAULTS['D']:.2f}, goalies {DEFAULTS['G']:.1f} games per 82")

Starter's share of the top two goalies' games, by season:
         median   mean    min    max  count
season                                     
2023-24   0.639  0.644  0.511  0.830     32
2024-25   0.682  0.661  0.514  0.803     32
2025-26   0.629  0.623  0.507  0.744     32

Used for 2026-27: starter 0.643 (54 of 84 games), backup 0.357 (30)

Rookie default (R1), 2023-24 to 2025-26: forwards 11.70 min/game, defensemen 16.03, goalies 3.0 games per 82


## 4. Projections for past seasons (what was known before each season)

To test the coach score honestly, each past season t needs the projection we would have made **before** it:

- **Skaters:** notebook 18's chosen setup (version a, 1,500 minutes). The prior is fit on the training seasons only, NHLe is refit through 2017-18, and the aging curve is refit without season t. A rookie with no NHL history gets the prior alone (his NHLe, draft, age), or his position's average if he has no data.
- **Goalies:** notebook 15's backtest projection (notebook 21's prior isn't rebuilt for past seasons, a limit). A rookie goalie gets the league average.

**Check:** on the test seasons this must give notebook 18's test error for version (a), 1,500 minutes exactly.

In [6]:
f_train, m_train = training_nhle(careers, TRAIN_END)
nhle_train = best_nhle(careers, f_train, m_train, rule="last2")
pairs = pd.concat([candidates(history, TARGETS + [2026]),
                   skaters[skaters["season"].isin(TARGETS)][["playerId", "season", "group"]]]).drop_duplicates(["playerId", "season"])
feat = skater_features(pairs, history, bios, profiles, nhle_train)
model = fit_prior(fit_rows(history, feat, TRAIN), PRIOR_INPUTS)
cand = candidates(history, TARGETS).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")
priors = cand[["playerId", "season"]].assign(prior_gar_60=apply_prior(cand, model).to_numpy())

rows = []
for t in TARGETS:
    curve = aging_curve(history, bios, exclude_season=t)[1]
    ps = project(skaters, bios, t, SK_WEIGHTS, SK_REG, AGING, curve, priors=priors, prior_reg=1500)
    pg = project(goalies, bios, t, G_WEIGHTS, G_REG, AGING, curve)
    rows.append(pd.concat([ps, pg])[["proj_war_60"]].assign(season=t).rename_axis("playerId").reset_index())
bt_proj = pd.concat(rows, ignore_index=True)

# Check against notebook 18's backtest (same players, same numbers)
b = backtest(skaters, TEST, SK_WEIGHTS, SK_REG, 300, bios, aging=AGING, priors=priors, prior_reg=1500)
m = b.merge(bt_proj, on=["playerId", "season"], how="left")
assert (m["proj_war_60"] * GOALS_PER_WIN - m["pred"]).abs().max() < 1e-12
b = b.merge(feat[["playerId", "season", "age"]], on=["playerId", "season"], how="left")
k = b["age"] > 0
err = np.sqrt(np.average((b["actual"][k] - b["pred"][k]) ** 2, weights=b["toi"][k]))
assert round(err, 4) == 0.2665, err
print(f"Projections for {len(TARGETS)} past seasons. Test-season error {err:.4f} GAR/60 = notebook 18's (a), 1,500 minutes.")

Projections for 14 past seasons. Test-season error 0.2665 GAR/60 = notebook 18's (a), 1,500 minutes.


## 5. Backtest: which w predicts real ice time best?

For every target season t (2009-10 to 2025-26, short seasons skipped as before) and every team:

1. **The pool** is the players who actually played **20+ games for that team in t** (goalies: 10+).
2. **Coach score** from their ice time in t-1 and their projection for t (section 4): nothing from season t itself.
3. **The slot rule** (center rule included) ranks them. Each rank gets that rank's minutes from the table of seasons t-3 to t-1; players past the 12th forward or 6th defenseman get the extras' minutes.
4. **Compare** with the ice time per game each player actually got in t.

**Measures:**

- the games-weighted error (RMSE) of ice time per game, forwards and defensemen;
- how often the predicted top-6 forwards and top-4 defensemen are the real ones;
- goalies: how often the predicted starter is the real one (most games), and the error in each goalie's share of games.

**Settings:** w = 0, 0.25, 0.5, 0.75, 1, and "usage first" (ice time rounded to the nearest minute decides; the rating breaks ties). Each comes with two rules for players without last-season ice time:

- **R1, rookie default:** treat him like a typical rookie (about a fourth-line forward or third-pair D); his projection can still move him up.
- **R2, expected from his projection:** a straight-line fit of ice time on projected WAR/60, so he's judged on his rating alone.

**A reference line:** everyone simply keeps last season's ice time (no slots, so the minutes don't have to add up to a game).

**What this test does and doesn't cover.** It predicts minutes **among the players who actually played** for each team. It does **not** test which players a coach dresses from the full pool (for example Chiarot vs. an AHL call-up), because the real season already made that choice. That selection is tested in step 2, against real standings.

In [7]:
pools = {}
for t in TARGETS:
    u = usage[usage["season"] == t]
    keep = np.where(u["group"] == "G", u["games_played"] >= 10, u["games_played"] >= L.MIN_GP_POOL)
    p = (u[keep][["playerId", "name", "team", "position", "group", "games_played", "usage"]]
         .rename(columns={"games_played": "gp", "usage": "actual"}).reset_index(drop=True))
    p["proj_war_60"] = p["playerId"].map(bt_proj[bt_proj["season"] == t].set_index("playerId")["proj_war_60"])
    p["proj_source"] = np.where(p["proj_war_60"].notna(), "projection", "")
    miss = p["proj_war_60"].isna() & (p["group"] != "G")            # skater rookies: the prior alone
    f = p.loc[miss, ["playerId", "group"]].merge(feat[feat["season"] == t].drop(columns="group"), on="playerId", how="left")
    p.loc[miss, "proj_war_60"] = (apply_prior(f, model) / GOALS_PER_WIN).to_numpy()
    p.loc[miss & p["proj_war_60"].notna(), "proj_source"] = "prior (rookie)"
    still = p["proj_war_60"].isna()                                # no data at all: position average
    p.loc[still, "proj_war_60"] = p.loc[still, "group"].map(group_average(history, t - 1) / GOALS_PER_WIN)
    p.loc[still, "proj_source"] = "position average"
    p["usage_last"] = L.last_usage(usage, p["playerId"], t)
    p["natural_center"] = L.natural_center(usage, p["playerId"], p["position"], t) & (p["group"] == "F")
    g = p["group"] == "G"                                          # a goalie's actual = his share of the pool's games
    p.loc[g, "actual"] = p.loc[g, "gp"] / p[g].groupby("team")["gp"].transform("sum")
    p["season"] = t
    seasons = [t - 3, t - 2, t - 1]
    pools[t] = {"pool": p, "defaults": L.rookie_defaults(usage, exp, seasons),
                "slots": L.slot_minutes(usage, seasons), "share": L.starter_share(usage, seasons)[1]}

allp = pd.concat([d["pool"] for d in pools.values()])
print(f"Backtest pools: {len(allp):,} player-seasons on {allp.groupby(['season', 'team']).ngroups} team-seasons")
print("Projection source:", allp["proj_source"].value_counts().to_dict())
print("Players without last-season ice time (rule R1/R2 applies):",
      allp[allp["usage_last"].isna()].groupby("group").size().to_dict())

Backtest pools: 10,616 player-seasons on 432 team-seasons
Projection source: {'projection': 9928, 'prior (rookie)': 629, 'position average': 59}
Players without last-season ice time (rule R1/R2 applies): {'D': 457, 'F': 897, 'G': 83}


In [8]:
SETTINGS = [(w, r) for w in [0, 0.25, 0.5, 0.75, 1, L.USAGE_FIRST] for r in ["R1", "R2"]]
res = []
for w, r in SETTINGS:
    for t, d in pools.items():
        sc = L.coach_scores(d["pool"], {g: (w, r) for g in "FDG"}, d["defaults"])
        lu = pd.concat([L.pick_lineup(x) for _, x in sc.groupby("team")])
        lu["pred"] = L.assign_minutes(lu, d["slots"], d["share"], col="raw", extras=True)
        lu["w"], lu["rule"] = str(w), r
        res.append(lu)
res = pd.concat(res, ignore_index=True)
res["period"] = np.where(res["season"] <= TRAIN_END, "train", "test")

def rmse(d, col="pred"):
    return float(np.sqrt(np.average((d[col] - d["actual"]) ** 2, weights=d["gp"])))

def top_match(d, g, k):
    """Average share of each team's real top-k (by actual ice time) that the slot rule also put in its top k."""
    out = [len(set(x.nlargest(k, "actual")["playerId"]) & set(x.loc[x["rank"] <= k, "playerId"])) / k
           for _, x in d[d["group"] == g].groupby(["season", "team"])]
    return float(np.mean(out))

def goalie_scores(d):
    x = d[d["group"] == "G"].copy()
    share = x["season"].map({t: p["share"] for t, p in pools.items()})
    x["pred"] = np.where(x["rank"] == 1, share, np.where(x["rank"] == 2, 1 - share, 0.0))
    starter = [x.loc[s["actual"].idxmax(), "rank"] == 1 for _, s in x.groupby(["season", "team"])]
    return rmse(x), float(np.mean(starter))

rows = []
for (w, r, per), d in res.groupby(["w", "rule", "period"], sort=False):
    g_err, g_start = goalie_scores(d)
    new = d["toi_rule"] != "own"
    rows.append({"period": per, "w": w, "rule": r,
                 "F error": rmse(d[d["group"] == "F"]), "D error": rmse(d[d["group"] == "D"]),
                 "top-6 F": top_match(d, "F", 6), "top-4 D": top_match(d, "D", 4),
                 "G share error": g_err, "G starter right": g_start,
                 "F error, no last-season TOI": rmse(d[(d["group"] == "F") & new]),
                 "D error, no last-season TOI": rmse(d[(d["group"] == "D") & new])})
grid = pd.DataFrame(rows)

ref = res[(res["w"] == "1") & (res["rule"] == "R1") & (res["group"] != "G")]
print("Errors are in minutes of ice time per game (goalies: share of games). Matches are shares of players.")
for per in ["test", "train"]:
    print(f"\n{per.capitalize()} seasons:")
    print(grid[grid["period"] == per].drop(columns="period").round(4).to_string(index=False))
    x = ref[ref["period"] == per]
    print(f"Reference, everyone keeps last season's ice time (R1 for the rest): "
          f"F {rmse(x[x['group'] == 'F'], 'toi_input'):.4f}, D {rmse(x[x['group'] == 'D'], 'toi_input'):.4f}")

Errors are in minutes of ice time per game (goalies: share of games). Matches are shares of players.

Test seasons:
          w rule  F error  D error  top-6 F  top-4 D  G share error  G starter right  F error, no last-season TOI  D error, no last-season TOI
          0   R1   2.7490   3.5247   0.6928   0.6715         0.2098           0.6073                       2.9022                       3.7904
          0   R2   2.7490   3.5247   0.6928   0.6715         0.2098           0.6073                       2.9022                       3.7904
       0.25   R1   2.3043   2.9907   0.7504   0.7330         0.1896           0.6649                       2.4763                       3.4418
       0.25   R2   2.3585   3.0448   0.7469   0.7356         0.1935           0.6702                       2.9118                       3.8338
        0.5   R1   1.9801   2.4631   0.7949   0.7971         0.1643           0.7382                       2.2254                       2.9373
        0.5   R2   2.0820 

In [9]:
COL = {"F": "F error", "D": "D error", "G": "G share error"}
CHOSEN, rows = {}, []
for g in ["F", "D", "G"]:
    te = grid[grid["period"] == "test"].set_index(["w", "rule"])[COL[g]]
    tr = grid[grid["period"] == "train"].set_index(["w", "rule"])[COL[g]]
    w, r = te.idxmin()
    CHOSEN[g] = (L.USAGE_FIRST if w == L.USAGE_FIRST else float(w), r)
    rows.append({"group": NAMES[g], "chosen (lowest test error)": f"w = {w}, {r}", "test error": f"{te.min():.6f}",
                 "best on training seasons": f"w = {tr.idxmin()[0]}, {tr.idxmin()[1]}",
                 "agrees": "yes" if tr.idxmin() == (w, r) else "no", "runner-up on test": f"w = {te.drop((w, r)).idxmin()[0]}, "
                 f"{te.drop((w, r)).idxmin()[1]} ({te.drop((w, r)).min():.6f})"})
print(pd.DataFrame(rows).to_string(index=False))
print("\nSaid plainly: these settings are picked on the test seasons (2018-19 on), as in notebook 18.")
print("The training seasons (through 2017-18) are shown next to them as the honest check.")
print("And again: this tests minutes among players who actually played for each team, not which players")
print("a coach dresses from the full pool (e.g. Chiarot vs. an AHL call-up). That's tested in step 2 against real standings.")
print("\nCoach-score settings:", CHOSEN)

     group chosen (lowest test error) test error best on training seasons agrees              runner-up on test
  Forwards               w = 0.75, R1   1.862451             w = 0.75, R1    yes w = usage first, R1 (1.894009)
Defensemen               w = 0.75, R1   2.187875             w = 0.75, R1    yes w = usage first, R1 (2.217050)
   Goalies                w = 0.5, R1   0.164278             w = 0.75, R1     no        w = 0.75, R1 (0.164283)

Said plainly: these settings are picked on the test seasons (2018-19 on), as in notebook 18.
The training seasons (through 2017-18) are shown next to them as the honest check.
And again: this tests minutes among players who actually played for each team, not which players
a coach dresses from the full pool (e.g. Chiarot vs. an AHL call-up). That's tested in step 2 against real standings.

Coach-score settings: {'F': (0.75, 'R1'), 'D': (0.75, 'R1'), 'G': (0.5, 'R1')}


**One more choice: how far back to look for usage.** The grid above uses last season only, as specified. But a player who missed last season (Aleksander Barkov was hurt for all of 2025-26) then gets the rookie default, and no coach would treat him like a rookie. So, at the chosen settings, the same backtest tries looking back **1, 2 or 3 seasons** for each player's most recent season with 10+ games (goalies: any games). That's still only information known before the season. The lowest test error wins, per group, with the training seasons shown as the check.

In [10]:
rows = []
for lb in [1, 2, 3]:
    res_lb = []
    for t, d in pools.items():
        p = d["pool"].assign(usage_last=L.last_usage(usage, d["pool"]["playerId"], t, lb))
        sc = L.coach_scores(p, CHOSEN, d["defaults"])
        lu = pd.concat([L.pick_lineup(x) for _, x in sc.groupby("team")])
        lu["pred"] = L.assign_minutes(lu, d["slots"], d["share"], col="raw", extras=True)
        res_lb.append(lu)
    r = pd.concat(res_lb, ignore_index=True)
    r["period"] = np.where(r["season"] <= TRAIN_END, "train", "test")
    for per in ["test", "train"]:
        x = r[r["period"] == per]
        rows.append({"lookback": lb, "period": per, "F error": rmse(x[x["group"] == "F"]),
                     "D error": rmse(x[x["group"] == "D"]), "G share error": goalie_scores(x)[0],
                     "players filled by a rule": int((x["toi_rule"] != "own").sum())})
lb_grid = pd.DataFrame(rows)
print("Errors at the chosen settings, by how many seasons back usage may come from:")
print(lb_grid.round(4).to_string(index=False))

LOOKBACK = {}
for g in ["F", "D", "G"]:
    te = lb_grid[lb_grid["period"] == "test"].set_index("lookback")[COL[g]]
    tr = lb_grid[lb_grid["period"] == "train"].set_index("lookback")[COL[g]]
    LOOKBACK[g] = int(te.idxmin())
    print(f"{NAMES[g]}: look back {LOOKBACK[g]} season(s): test error {te.min():.6f} (last season only: {te.loc[1]:.6f}). "
          f"Best on the training seasons: {tr.idxmin()} ({'agrees' if tr.idxmin() == LOOKBACK[g] else 'differs'})")
print("\nSettings used for 2026-27: coach score", CHOSEN, "| lookback", LOOKBACK)

Errors at the chosen settings, by how many seasons back usage may come from:
 lookback period  F error  D error  G share error  players filled by a rule
        1   test   1.8625   2.1879         0.1643                       589
        1  train   2.0755   2.3698         0.1986                       848
        2   test   1.8410   2.1917         0.1638                       477
        2  train   2.0812   2.3621         0.1981                       724
        3   test   1.8430   2.1988         0.1638                       446
        3  train   2.0792   2.3623         0.1984                       706
Forwards: look back 2 season(s): test error 1.840958 (last season only: 1.862451). Best on the training seasons: 1 (differs)
Defensemen: look back 1 season(s): test error 2.187875 (last season only: 2.187875). Best on the training seasons: 2 (differs)
Goalies: look back 2 season(s): test error 0.163801 (last season only: 0.164278). Best on the training seasons: 2 (agrees)

Settings used f

## 6. 2026-27 opening-night lineups

**Each team's pool** is its opening-night NHL level (active roster, off the active roster, and the two disputes) plus its AHL/ECHL players on NHL contracts (call-ups). Not unsigned draftees in junior, college or Europe, not AHL-contract players.

**Usage** is ice time per game in his most recent season with 10+ games within the chosen lookback (normally 2025-26), and the projection is notebook 22's `current_projections()`. Coach scores are computed once on the whole league pool, so a trade or a sit-out doesn't change anyone's score, only who's on the team.

**Natural centers:** 20+ faceoffs per 60 in a player's last NHL season (10+ games), otherwise his listed position. Faceoffs are part of WAR, so every lineup needs 4 centers:

- If the top 12 have fewer, the best center outside them replaces the lowest-ranked winger.
- Only if no center is left does a winger play center (flagged).

**Baseline:** no trades applied, disputes play.

**Lineup strength** = the sum of projected WAR/60 x minutes per game / 60 over the 20 dressed players, i.e. **WAR per game** (also per 84 games, the 2026-27 schedule). It's a preview, not standings: no schedule, no luck, no injuries.

In [11]:
players = L.load_players(DATA, usage, current_projections(DATA), LOOKBACK)
pool = players[players["in_pool"]]
assert pool["proj_war_60"].notna().all() and pool["group"].notna().all()
print(f"Pool: {len(pool):,} players on {pool['team'].nunique()} teams "
      f"({(pool['group'] == 'F').sum()} F, {(pool['group'] == 'D').sum()} D, {(pool['group'] == 'G').sum()} G)")
print("By status:", pool["status"].value_counts().to_dict())
sizes = pool.groupby(["team", "group"]).size().unstack()
centers = pool[pool["natural_center"]].groupby("team").size()
print(f"Per team: F {sizes['F'].min()}-{sizes['F'].max()}, D {sizes['D'].min()}-{sizes['D'].max()}, "
      f"G {sizes['G'].min()}-{sizes['G'].max()}, natural centers {centers.min()}-{centers.max()}")

scored = L.coach_scores(pool, CHOSEN, DEFAULTS)
print("\nWho got which usage rule:", scored.groupby(["group", "toi_rule"]).size().unstack(fill_value=0).to_dict("index"))
older = scored[scored["usage_last"].notna() & np.isnan(L.last_usage(usage, scored["playerId"], 2026, 1))]
print(f"Usage from an older season (missed most of 2025-26), {len(older)} players. Highest projected:")
print(older.nlargest(8, "proj_war_60")[["name", "team", "group", "status", "usage_last", "proj_war_60"]].round(3).to_string(index=False))
trades = pd.read_csv(TRADES_PATH)
base = L.build_lineups(scored, slots, SHARE)
dressed = base[base["dressed"]]
count = dressed.groupby("team").agg(F=("group", lambda x: (x == "F").sum()), D=("group", lambda x: (x == "D").sum()),
                                    G=("group", lambda x: (x == "G").sum()), C=("role", lambda x: (x == "C").sum()))
assert (count["F"] == 12).all() and (count["D"] == 6).all() and (count["G"] == 2).all()
assert dressed["proj_war_60"].notna().all()
print(f"Every team dresses 12 F, 6 D, 2 G. Teams with 4 centers: {(count['C'] == 4).sum()} of {len(count)}")
flags = base[base["flag"] != ""]
print("\nFlagged:" if len(flags) else "\nNothing flagged.")
if len(flags):
    print(flags[["team", "name", "slot", "flag"]].to_string(index=False))

Pool: 1,348 players on 32 teams (776 F, 433 D, 139 G)
By status: {'NHL roster': 797, 'AHL/ECHL (NHL contract)': 526, 'NHL org, off active roster': 23, 'suspended / holdout': 1, 'unsigned RFA': 1}
Per team: F 21-28, D 10-17, G 3-7, natural centers 5-13

Who got which usage rule: {'D': {'R1': 182, 'own': 251}, 'F': {'R1': 265, 'own': 511}, 'G': {'R1': 36, 'own': 103}}
Usage from an older season (missed most of 2025-26), 39 players. Highest projected:
             name team group                  status  usage_last  proj_war_60
Aleksander Barkov  FLA     F              NHL roster      20.258        0.087
  Philip Tomasino  OTT     F AHL/ECHL (NHL contract)      13.066        0.071
       Joshua Roy  UTA     F AHL/ECHL (NHL contract)      11.018        0.061
  Jakob Pelletier  TBL     F AHL/ECHL (NHL contract)      11.556        0.059
       Devon Levi  EDM     G              NHL roster       9.000        0.058
  Matthew Poitras  BOS     F              NHL roster      13.088        0.057
 

In [12]:
def show(lu, team):
    """One team's lineup: lines, pairs and goalies with minutes per game."""
    d = lu[(lu["team"] == team) & lu["dressed"]]
    fmt = lambda r: f"{r['name']} ({r['minutes_per_game']:.1f})"
    out = []
    for k in range(1, 5):
        ln = d[(d["group"] == "F") & (d["line"] == k)]
        c, wings = ln[ln["role"] == "C"], ln[ln["role"] == "W"].sort_values("rank")
        out.append(f"  L{k}  " + "  |  ".join([fmt(r) for _, r in wings.iloc[:1].iterrows()] + [fmt(r) for _, r in c.iterrows()]
                                           + [fmt(r) for _, r in wings.iloc[1:].iterrows()]))
    for k in range(1, 4):
        pr = d[(d["group"] == "D") & (d["line"] == k)].sort_values("rank")
        out.append(f"  P{k}  " + "  |  ".join(fmt(r) for _, r in pr.iterrows()))
    gl = d[d["group"] == "G"].sort_values("rank")
    out.append("  G   " + "  |  ".join(f"{r['name']} ({r['minutes_per_game'] / 60 * 84:.0f} GP)" for _, r in gl.iterrows()))
    return "\n".join(out)

strength = L.lineup_strength(base)
for team in strength.index:
    print(f"{team}  {strength.at[team, 'war_per_84']:.1f} WAR per 84 games")
    print(show(base, team))

WSH  28.9 WAR per 84 games
  L1  Alex Tuch (19.3)  |  Dylan Strome (16.7)  |  Tom Wilson (18.4)
  L2  Aliaksei Protas (17.5)  |  Pierre-Luc Dubois (14.7)  |  Jordan Kyrou (16.0)
  L3  Alex Ovechkin (15.4)  |  Boone Jenner (13.9)  |  Ryan Leonard (13.2)
  L4  Anthony Beauvillier (12.4)  |  Justin Sourdif (11.9)  |  Andrew Cristall (11.2)
  P1  Jakob Chychrun (23.4)  |  Matt Roy (21.4)
  P2  Cole Hutson (20.0)  |  Rasmus Sandin (18.7)
  P3  Timothy Liljegren (17.1)  |  Martin Fehérváry (15.8)
  G   Logan Thompson (54 GP)  |  Charlie Lindgren (30 GP)
EDM  27.2 WAR per 84 games
  L1  Zach Hyman (17.5)  |  Connor McDavid (19.3)  |  Ryan Nugent-Hopkins (16.7)
  L2  Vasily Podkolzin (15.4)  |  Leon Draisaitl (18.4)  |  Matt Savoie (14.7)
  L3  Kasperi Kapanen (13.9)  |  Jason Dickinson (16.0)  |  Alex Formenton (13.2)
  L4  Viljami Marjala (12.4)  |  Connor Clattenburg (11.2)  |  Mathieu Joseph (11.9)
  P1  Evan Bouchard (23.4)  |  Mattias Ekholm (21.4)
  P2  Ryan Shea (20.0)  |  Shakir Mukha

In [13]:
st = strength.copy()
st.insert(0, "rank", np.arange(1, len(st) + 1))
best = dressed.sort_values("war_per_game", ascending=False).groupby("team")["name"].apply(lambda x: ", ".join(x.iloc[:3]))
st["top 3 by WAR"] = best
print("Lineup strength, WAR per game by group (preview, not standings):")
print(st.round(3).to_string())

Lineup strength, WAR per game by group (preview, not standings):
group  rank      F      D      G  war_per_game  war_per_84                                        top 3 by WAR
team                                                                                                          
WSH       1  0.205  0.072  0.068         0.345      28.949        Logan Thompson, Jordan Kyrou, Jakob Chychrun
EDM       2  0.208  0.072  0.044         0.324      27.241       Connor McDavid, Leon Draisaitl, Tristan Jarry
BUF       3  0.194  0.077  0.052         0.322      27.079  Tage Thompson, Rasmus Dahlin, Ukko-Pekka Luukkonen
DAL       4  0.216  0.047  0.057         0.320      26.907     Jason Robertson, Wyatt Johnston, Jake Oettinger
VGK       5  0.199  0.057  0.064         0.320      26.861              Carter Hart, Jack Eichel, Mitch Marner
MTL       6  0.186  0.071  0.060         0.317      26.641             Cole Caufield, Jakub Dobes, Nick Suzuki
CAR       7  0.209  0.064  0.041         0.314 

In [14]:
# Hockey check 1: last season. Each team's 2025-26 total WAR (skaters + goalies, by final team)
last = history[history["season"] == 2025].groupby("team")["war"].sum()
last.index = last.index.map(lambda x: "UTA" if x == "ARI" else x)
chk = strength[["war_per_84"]].join(last.rename("war_2025_26"))
print(f"Correlation of 2026-27 lineup strength with 2025-26 team WAR: {chk['war_per_84'].corr(chk['war_2025_26']):.2f}")

# Hockey check 2: the real 2026-27 standings so far (context only, never used to fit anything)
standings = standings_table(download_standings(TODAY, DATA / "current_raw" / f"standings_{TODAY}.json"))
chk = chk.join(standings.set_index("team"))
r = chk["war_per_84"].corr(chk["point_pct"])
print(f"Correlation with real 2026-27 points percentage as of {TODAY}: {r:.2f} "
      f"(teams have played {chk['games_played'].min()}-{chk['games_played'].max()} games, so this is mostly noise; context only)")
chk["strength rank"] = chk["war_per_84"].rank(ascending=False).astype(int)
chk["points % rank"] = chk["point_pct"].rank(ascending=False, method="min").astype(int)
print("\nTop 5 and bottom 5 by lineup strength, with last season and this season so far:")
print(pd.concat([chk.head(5), chk.tail(5)])[["war_per_84", "war_2025_26", "games_played", "point_pct", "strength rank",
                                             "points % rank"]].round(3).to_string())

Correlation of 2026-27 lineup strength with 2025-26 team WAR: 0.64
Correlation with real 2026-27 points percentage as of 2026-11-06: 0.48 (teams have played 14-18 games, so this is mostly noise; context only)

Top 5 and bottom 5 by lineup strength, with last season and this season so far:
      war_per_84  war_2025_26  games_played  point_pct  strength rank  points % rank
team                                                                                
WSH       28.949       24.597            14      0.571              1             16
EDM       27.241       24.528            17      0.735              2              2
BUF       27.079       28.090            15      0.567              3             17
DAL       26.907       26.944            16      0.625              4             10
VGK       26.861       18.740            18      0.667              5              8
LAK       20.667       14.036            14      0.429             28             28
PHI       20.404       10.194 

## 7. Ryan's experiment, first look: Detroit, and the Hellebuyck trade

Detroit four ways: (a) as is, (b) Larkin sits, (c) Edvinsson sits, (d) both sit. For each, who moves into the open slots and what it costs in WAR per 84 games. Then Winnipeg and New Jersey with and without trade T1 (Hellebuyck to New Jersey for Mercer, Daws, Bjugstad and the rights to Command).

This is lineup strength only, not wins yet. Step 2 turns WAR into goals and wins.

In [15]:
pid = lambda name: int(players.loc[players["name"] == name, "playerId"].iloc[0])
LARKIN, EDVINSSON = pid("Dylan Larkin"), pid("Simon Edvinsson")
assert set(L.dispute_ids(players)) == {LARKIN, EDVINSSON}

def changes(before, after, team):
    """Slot by slot: who held it before and after, for every slot that changed."""
    b = before[(before["team"] == team) & before["dressed"]].set_index("slot")
    a = after[(after["team"] == team) & after["dressed"]].set_index("slot")
    rows = [{"slot": s, "before": b.at[s, "name"], "min before": b.at[s, "minutes_per_game"],
             "after": a.at[s, "name"], "min after": a.at[s, "minutes_per_game"]}
            for s in b.index if b.at[s, "playerId"] != a.at[s, "playerId"]]
    return pd.DataFrame(rows)

det = {"(a) as is": base,
       "(b) Larkin sits": L.build_lineups(scored, slots, SHARE, sit_out=[LARKIN]),
       "(c) Edvinsson sits": L.build_lineups(scored, slots, SHARE, sit_out=[EDVINSSON]),
       "(d) both sit": L.build_lineups(scored, slots, SHARE, sit_out=[LARKIN, EDVINSSON])}
summary = []
for k, lu in det.items():
    s = L.lineup_strength(lu)
    summary.append({"scenario": k, "WAR per 84": s.at["DET", "war_per_84"],
                    "change": s.at["DET", "war_per_84"] - strength.at["DET", "war_per_84"],
                    "league rank": int((s["war_per_84"] > s.at["DET", "war_per_84"]).sum() + 1)})
print("Detroit lineup strength:")
print(pd.DataFrame(summary).round(2).to_string(index=False))
print("\n(a) Detroit as is:")
print(show(base, "DET"))
for k in ["(b) Larkin sits", "(c) Edvinsson sits", "(d) both sit"]:
    print(f"\n{k}: who moves up")
    print(changes(base, det[k], "DET").round(1).to_string(index=False))

Detroit lineup strength:
          scenario  WAR per 84  change  league rank
         (a) as is       21.09    0.00           27
   (b) Larkin sits       19.59   -1.50           29
(c) Edvinsson sits       20.92   -0.18           27
      (d) both sit       19.42   -1.68           29

(a) Detroit as is:
  L1  Lucas Raymond (18.4)  |  Dylan Larkin (19.3)  |  Alex DeBrincat (17.5)
  L2  Emmitt Finnie (16.7)  |  Andrew Copp (16.0)  |  Viktor Arvidsson (15.4)
  L3  Mason Appleton (13.2)  |  J.T. Compher (14.7)  |  Michael Rasmussen (12.4)
  L4  Michael Brandsegg-Nygård (11.9)  |  Marco Kasper (13.9)  |  Carter Bear (11.2)
  P1  Moritz Seider (23.4)  |  Simon Edvinsson (21.4)
  P2  Justin Faulk (20.0)  |  Ben Chiarot (18.7)
  P3  Axel Sandin-Pellikka (17.1)  |  Anton Johansson (15.8)
  G   John Gibson (54 GP)  |  Daniil Tarasov (30 GP)

(b) Larkin sits: who moves up
slot                   before  min before                    after  min after
  F1             Dylan Larkin        19.3       

In [16]:
t1 = L.build_lineups(scored, slots, SHARE, trades, trade_ids=["T1"])
s1 = L.lineup_strength(t1)
rows = [{"team": tm, "without T1": strength.at[tm, "war_per_84"], "with T1": s1.at[tm, "war_per_84"],
         "change": s1.at[tm, "war_per_84"] - strength.at[tm, "war_per_84"]} for tm in ["WPG", "NJD"]]
print("Trade T1 (Oct 23):", "; ".join(f"{r['name']} {r['from_team']}->{r['to_team']}" for _, r in trades[trades["trade_id"] == "T1"].iterrows()))
print(pd.DataFrame(rows).round(2).to_string(index=False))
for tm in ["WPG", "NJD"]:
    print(f"\n{tm}, slots that change with T1:")
    print(changes(base, t1, tm).round(1).to_string(index=False))

Trade T1 (Oct 23): Connor Hellebuyck WPG->NJD; Dawson Mercer NJD->WPG; Nico Daws NJD->WPG; Nick Bjugstad NJD->WPG; Alexander Command NJD->WPG
team  without T1  with T1  change
 WPG       22.53    21.34   -1.19
 NJD       24.85    26.55    1.71

WPG, slots that change with T1:
slot                before  min before                 after  min after
  F4         Cole Perfetti        16.7         Dawson Mercer       16.7
  F5          Alex Iafallo        16.0         Cole Perfetti       16.0
  F6            Adam Lowry        15.4          Alex Iafallo       15.4
  F7     Nino Niederreiter        14.7            Adam Lowry       14.7
  F8         Morgan Barron        13.9     Nino Niederreiter       13.9
  F9 Vladislav Namestnikov        13.2         Morgan Barron       13.2
 F10            Isak Rosen        12.4 Vladislav Namestnikov       12.4
 F11         Brayden Yager        11.9            Isak Rosen       11.9
 F12          Viggo Björck        11.2         Brayden Yager       11.2
  G

## 8. Chiarot check

Where Detroit's coach puts Ben Chiarot, and how many minutes he gets, vs. how he was really used in 2025-26.

In [17]:
ch = base[base["name"] == "Ben Chiarot"].iloc[0]
d26 = base[(base["team"] == "DET") & (base["group"] == "D")].sort_values("rank")
print(f"Ben Chiarot: coach score {ch['coach_score']:.2f}, rank {ch['rank']} of {len(d26)} Detroit D, slot {ch['slot']}, "
      f"{ch['minutes_per_game']:.2f} min/game")
real = usage[(usage["season"] == 2025) & (usage["team"] == "DET") & (usage["group"] == "D") & (usage["games_played"] >= 20)]
real = real.sort_values("usage", ascending=False).reset_index(drop=True)
rk = real.index[real["name"] == "Ben Chiarot"][0] + 1
print(f"2025-26 for real: {real.at[rk - 1, 'usage']:.2f} min/game, rank {rk} of {len(real)} Detroit D with 20+ games")
print("\nDetroit D, 2026-27 coach order:")
print(d26[["rank", "slot", "name", "status", "toi_input", "toi_rule", "proj_war_60", "coach_score", "minutes_per_game",
           "war_per_84"]].round(3).to_string(index=False))
print("\nDetroit D, 2025-26 real ice time (20+ games):")
print(real[["name", "games_played", "usage"]].rename(columns={"usage": "min/game"}).round(2).to_string(index=False))

Ben Chiarot: coach score 0.18, rank 4 of 14 Detroit D, slot D4, 18.74 min/game
2025-26 for real: 20.84 min/game, rank 4 of 8 Detroit D with 20+ games

Detroit D, 2026-27 coach order:
 rank  slot                 name                  status  toi_input toi_rule  proj_war_60  coach_score  minutes_per_game  war_per_84
    1    D1        Moritz Seider              NHL roster     25.666      own        0.047        2.389            23.429       1.537
    2    D2      Simon Edvinsson            unsigned RFA     22.356      own        0.026        1.280            21.438       0.779
    3    D3         Justin Faulk              NHL roster     22.010      own        0.014        1.018            20.024       0.379
    4    D4          Ben Chiarot              NHL roster     20.837      own       -0.025        0.185            18.739      -0.650
    5    D5 Axel Sandin-Pellikka AHL/ECHL (NHL contract)     16.211      own        0.032       -0.139            17.145       0.771
    6    D6      An

## 9. Save

`data/lineups_2026-27.csv` (gitignored): every pool player on every team, baseline (no trades applied, disputes play). Scenarios (sit-outs, trades) are rebuilt from `lineups.py` when needed.

In [18]:
COLS = ["team", "playerId", "name", "group", "natural_center", "status", "dressed", "slot", "line", "role", "coach_score",
        "toi_input", "toi_rule", "proj_war_60", "minutes_per_game", "war_per_game", "war_per_84", "flag"]
save = base.sort_values(["team", "group", "rank"])[COLS]
save.to_csv(OUT / "lineups_2026-27.csv", index=False)
print(f"Saved lineups_2026-27.csv: {len(save):,} players on {save['team'].nunique()} teams, {int(save['dressed'].sum())} dressed")

Saved lineups_2026-27.csv: 1,348 players on 32 teams, 640 dressed


## Takeaways
- **The trade log has five more trades** (T2 to T6, 6 rows), all built from the game logs:
  - Hallander PIT to EDM and Struble MTL to NSH (Oct 10)
  - Hunt MIN to NSH (Oct 25)
  - St. Ivany WPG to WSH (Oct 28)
  - Shane Wright and Matthew Wood, one SEA-NSH swap (Nov 3)

  The baseline applies none of them.
- **Minutes per slot (2023-24 to 2025-26):** a team's forwards share 180.6 minutes a game and its defensemen 116.6. Scaled to those totals:
  - forwards run from 19.3 (1st) to 11.2 (12th)
  - defensemen from 23.4 (1st) to 15.8 (6th)

  The raw medians add up to more (189.1 and 120.3), because injury fill-ins also count.
- **Goalies:** the starter gets 64.3% of the top two goalies' games, so 54 of 84 games, and the backup 30. Back-to-backs come in the season step.
- **Rookie default (R1):** forwards 11.7 minutes, defensemen 16.0, goalies 3 games per 82.
- **The past-season projections reproduce notebook 18 exactly** (test error 0.2665).
- **The backtest:** 10,616 player-seasons on 432 team-seasons from 2009-10 to 2025-26.
  - **The winner for forwards and defensemen is w = 0.75 with the rookie default (R1):** three parts real usage, one part projection.
    - Forwards: test error 1.86 minutes per game. Defensemen: 2.19.
    - The training seasons pick the same setting.
    - It gets 82% of each team's real top-6 forwards and 81% of its top-4 D right.
  - **Projection only (w = 0) is clearly worst** (forwards 2.75). Usage alone (w = 1) is 1.94, and "usage first, rating breaks ties" is 1.89. Coaches mostly keep using who they used, and the projection adds a little.
  - **R1 has the lower forward and defense error than R2 in every setting,** test and training. Treating an unknown like a typical rookie predicts better than trusting his projection. On the test seasons, the players without last season's ice time are where R1 gains the most.
  - **Goalies: w = 0.5 (R1) by a hair.** The test error is 0.164278 vs. 0.164283 for w = 0.75, and the training seasons prefer 0.75, so it's a coin flip. The predicted starter is the real one 74% of the time.
  - **Lookback:** forwards and goalies look back 2 seasons, defensemen 1. The differences are tiny on average (forwards 1.841 vs. 1.862), and the training seasons disagree for forwards and defensemen. It matters for a few players: 39 in 2026-27 have usage from an older season. The biggest is Barkov, who's back as Florida's second-line center (18.4 minutes) instead of a rookie's role.
  - **Said plainly:**
    - All settings are picked on the test seasons, as in notebook 18.
    - This tests minutes **among players who actually played for each team**, not which players a coach dresses from the full pool (Chiarot vs. an AHL call-up). That's tested in step 2, against real standings.
  - **Honest finding: giving each player his own last-season minutes is slightly more accurate** (forwards 1.81, D 2.11) than slot minutes, but those minutes don't add up to a real game.
    - Slot minutes flatten the stars: McDavid gets 19.3 here, not his real 23.0.
    - A blend of slot minutes and a player's own minutes is a candidate for later.
- **2026-27 opening-night lineups:**
  - **The pool:** 1,348 players (797 on active rosters, 526 on NHL contracts in the AHL/ECHL, 23 off the active roster, and the two disputes).
  - **Every team dresses 12 F, 6 D and 2 G with 4 natural centers.** The center rule brought in a center at 3 teams: EDM (Clattenburg), LAK (Pederson) and NJD (Lombardi). No winger had to play center.
  - **Strength preview** (WAR per 84 games, not standings):
    - Top: WSH 28.9, EDM 27.2, BUF 27.1, DAL 26.9, VGK 26.9
    - Bottom: CGY 15.9, SEA 18.6, NYI 19.0, PHI 20.4, LAK 20.7
    - Detroit is 27th (21.1).
  - **Hockey check:**
    - It correlates 0.64 with each team's 2025-26 total WAR.
    - It correlates 0.48 with the real 2026-27 points percentage so far. That's only 14 to 18 games, so it's context, not proof.
    - Edmonton is 2nd in both. Seattle (31st in strength, 9th in points %) is the biggest early surprise.
- **Ryan's experiment, first look:**
  - **Larkin sits:** Detroit loses 1.50 WAR per 84 games (27th to 29th). Every forward moves up a slot and Jesse Kiiskinen dresses.
  - **Edvinsson sits:** only -0.18. Faulk moves to D2, Chiarot to D3, and Shai Buium dresses. Edvinsson's projection (0.026 WAR/60) is close to the prospects who replace him.
  - **Both sit:** -1.68.
  - **The Hellebuyck trade (T1):**
    - Winnipeg -1.19 (Skinner becomes the starter and Daws the backup; Mercer slots in as the 4th forward).
    - New Jersey +1.71 (Hellebuyck takes over from Allen).
- **Chiarot:** Detroit's coach puts him 4th of 14 D (slot D4, 18.7 minutes per game).
  - In 2025-26 he really was 4th of 8 (20.8 minutes).
  - His projection is the lowest of Detroit's top six D (-0.025 WAR/60, -0.65 WAR per 84). His usage keeps him in, as w = 0.75 says it should.
- **A quirk to watch:** the rookie default for defensemen (16.0 minutes) is a little higher than Albert Johansson's real 15.9. So the 82-game regular sits behind three prospects with the default, a selection question this backtest can't check.
- **Limits:**
  - A traded player's 2025-26 usage is his combined season.
  - Handedness isn't modeled.
  - Goalies' past projections don't include notebook 21's prior.
  - Lineups are fixed at the start, with full health.
- **Files:** `data/lineups_2026-27.csv` (gitignored): 1,348 pool players and 640 dressed, baseline only.
- **Next (step 2):** goals and wins from lineup WAR, back-to-backs, and the season simulator, tested against real standings.